# Цель работы

Решить задачу о назначениях с помощью бинарной сети Хопфилда.

# Задачи

Для матрицы стоимостей найти бинарную матрицу назначений: в каждой строке и каждом столбце ровно одна единица. Минимизировать суммарную стоимость; проверить допустимость решения и сравнить с полным перебором для малого примера.

# Ход работы

Энергия сети включает стоимость и квадратичные штрафы:

$$E(X)=\sum_{ij}c_{ij}x_{ij}+A\sum_i(\sum_j x_{ij}-1)^2+B\sum_j(\sum_i x_{ij}-1)^2.$$

При бинарных состояниях это квадратичная функция энергии сети Хопфилда. Нейроны обновляются асинхронно в случайном порядке: изменение принимается только при строгом уменьшении энергии. Штрафы выбираются больше суммы стоимостей. Для выхода из локальных минимумов выполняется несколько независимых запусков с фиксированным генератором. Глобальная оптимальность для произвольной матрицы не гарантируется.


In [1]:
import numpy as np
from itertools import permutations

class HopfieldNetwork:
    def __init__(self, costs, seed=42):
        self.costs = np.asarray(costs, dtype=float)
        if (self.costs.ndim != 2 or self.costs.shape[0] != self.costs.shape[1]
                or self.costs.size == 0 or not np.isfinite(self.costs).all()
                or (self.costs < 0).any()):
            raise ValueError("Ожидается непустая квадратная неотрицательная матрица")
        self.n = len(self.costs)
        self.penalty = float(self.costs.sum() + 1)
        self.rng = np.random.default_rng(seed)

    def energy(self, x):
        return float((x*self.costs).sum() + self.penalty * (
            ((x.sum(0)-1)**2).sum() + ((x.sum(1)-1)**2).sum()))

    @staticmethod
    def feasible(x):
        return (np.isin(x,[0,1]).all() and (x.sum(0)==1).all()
                and (x.sum(1)==1).all())

    def solve(self, restarts=200):
        best, best_energy, best_history = None, float("inf"), None
        for _ in range(restarts):
            x = self.rng.integers(0,2,size=(self.n,self.n))
            energy = self.energy(x)
            history = [energy]
            for sweep in range(100):
                changed = False
                for idx in self.rng.permutation(self.n*self.n):
                    i,j = divmod(int(idx),self.n)
                    x[i,j] = 1-x[i,j]
                    candidate = self.energy(x)
                    if candidate < energy - 1e-10:
                        energy = candidate
                        history.append(energy)
                        changed = True
                    else:
                        x[i,j] = 1-x[i,j]
                if not changed: break
            if self.feasible(x) and energy < best_energy:
                best, best_energy, best_history = x.copy(), energy, history
        if best is None:
            raise RuntimeError("Допустимое решение не найдено: увеличьте число запусков")
        return best, best_history


In [2]:
costs = np.array([[2,3,1],[5,1,4],[3,2,2]])
network = HopfieldNetwork(costs)
solution, history = network.solve()
print("Матрица стоимостей:\n",costs)
print("Матрица назначений:\n",solution)
print("Суммарная стоимость:", (solution*costs).sum())
print("Энергия выбранного запуска:",history)
assert network.feasible(solution)
assert np.all(np.diff(history) < 0)
exact_cost = min(sum(costs[i,j] for i,j in enumerate(p))
                 for p in permutations(range(len(costs))))
print("Минимум по полному перебору:",exact_cost)
assert (solution*costs).sum() == exact_cost == 5
# Additional matrices exercise both feasibility and objective evaluation.
for matrix in [np.array([[7]]), np.array([[9,1],[2,8]]), np.zeros((3,3))]:
    net = HopfieldNetwork(matrix)
    x,_ = net.solve()
    assert net.feasible(x)
    optimum = min(sum(matrix[i,j] for i,j in enumerate(p))
                  for p in permutations(range(len(matrix))))
    assert (x*matrix).sum() == optimum
print("Дополнительные проверки пройдены.")


Матрица стоимостей:
 [[2 3 1]
 [5 1 4]
 [3 2 2]]
Матрица назначений:
 [[1 0 0]
 [0 1 0]
 [0 0 1]]
Суммарная стоимость: 5
Энергия выбранного запуска: [107.0, 56.0, 5.0]
Минимум по полному перебору: 5
Дополнительные проверки пройдены.


# Выводы

Реализована бинарная сеть Хопфилда с асинхронным уменьшением энергии и случайными перезапусками. Для исходной матрицы найдено допустимое назначение стоимости 5, совпадающее с результатом полного перебора. Проверены ограничения по строкам и столбцам, строгое убывание энергии и дополнительные малые матрицы. Для больших задач метод остаётся эвристическим.
